# bibr 🦫 — Python Library Demo

Use `await bibr.achew_file(path)` inside Jupyter to extract one scientific paper.
It returns a `Result` with a validated export model and the JSON dictionary in
`result.data`. In a synchronous Python script, use `bibr.chew_file(path)`.

**Prerequisites:**
- Install bibr using the [installation guide](../docs/getting-started/install.md).
- Run `bibr setup` to configure OCR, models, and any required provider credentials.
- This notebook uses the packaged synthetic sample. Set `PAPER_PATH` before
  starting Jupyter to process your own document instead.


## 1. Process a single paper

In [ ]:
import os
from importlib.resources import as_file, files
from pathlib import Path

import bibr

configured_path = os.getenv("PAPER_PATH")
if configured_path:
    paper_path = Path(configured_path).expanduser()
    result = await bibr.achew_file(paper_path)
else:
    with as_file(files("bibr").joinpath("data/sample_paper.pdf")) as sample_path:
        result = await bibr.achew_file(sample_path)

data = result.data
print("Top-level keys:", list(data.keys()))


## 2. Inspect extracted metadata

`result.data` matches the bibr JSON export schema. `metadata` holds paper-level
metadata; `author`, `bib`, `section`, `text`, `eq`, `xref`, `bib_match`,
`figure`, `table`, and `url` are flat tables.


In [ ]:
metadata = data["metadata"]
print(f"Title:      {metadata['title']}")
print(f"DOI:        {metadata['doi']}")
print(f"Paper type: {metadata.get('paper_type')} ({(metadata.get('paper_type_confidence') or 0):.0%})")
print(f"OECD:       {metadata.get('oecd_l1')} → {metadata.get('oecd_l2')} ({(metadata.get('oecd_confidence') or 0):.0%})")
print(f"Keywords:   {metadata.get('keywords')}")
print(f"Authors:    {len(data['author'])}")
print(f"Refs:       {len(data['bib'])}")
print(f"Equations:  {len(data['eq'])}")
print(f"Figures:    {len(data['figure'])}")
print(f"Tables:     {len(data['table'])}")

### Authors

In [ ]:
for a in data["author"]:
    line = f"  {a.get('given', '')} {a.get('family', '')}"
    if a.get("affiliation"):
        line += f" — {a['affiliation']}"
    if a.get("corresponding"):
        line += f" (corresponding, {a.get('email')})"
    print(line)

### Sections (IMRaD classification)

In [ ]:
# How each section's type was decided lives under extraction.diagnostics.
diagnostics = (data.get("extraction") or {}).get("diagnostics") or {}
scores = {row["section_id"]: row["score"] for row in diagnostics.get("section_classification") or []}
for s in data["section"]:
    indent = "  " * (s.get("level", 1) - 1)
    score = scores.get(s["section_id"])
    score_text = "unscored" if score is None else f"{score:.0%}"
    header = s.get("header") or "(no heading)"
    print(f"{indent}[{s.get('section_type') or 'unknown':>15}] {header}  ({score_text})")

## 3. Work with DataFrames

Every flat table can be loaded into pandas directly.

In [ ]:
import pandas as pd

text_df = pd.DataFrame(data["text"])
print(f"Sentences: {len(text_df)}")
text_df.reindex(columns=["text_id", "section_id", "text"]).head(10)


### Equations

bibr extracts statistical test results and LaTeX equations, decomposed into
left-hand side, comparator, and right-hand side components.

In [ ]:
pd.DataFrame(data["eq"]).head()

### Tables

In [ ]:
for table in data["table"]:
    print(f"Table {table.get('label') or table['table_id']}: {table.get('caption') or '(no caption)'}")
    if table.get("contents"):
        display(pd.DataFrame(table["contents"]))


### References

In [ ]:
for ref in data["bib"][:10]:
    print(f"  [{ref['bib_id']}] {ref.get('authors', '')} ({ref.get('year', '?')}). {ref.get('title', '')}")

if len(data["bib"]) > 10:
    print(f"  ... and {len(data['bib']) - 10} more")

### Crossref reference matches

When `CROSSREF_ENRICH=true` (default), parsed references are matched against
Crossref. Matches land in the flat `bib_match` table keyed by `bib_id`.

In [ ]:
matches = pd.DataFrame(data["bib_match"])
if len(matches):
    crossref_matches = matches[matches["service"] == "crossref"]
    print(f"Crossref matched {len(crossref_matches)}/{len(data['bib'])} references")
    display(crossref_matches[["bib_id", "score", "title", "doi"]].head(10))
else:
    print("No matches found (enrichment may be disabled)")


## 4. Batch processing

Use an async `Chewer` context to reuse a pipeline across calls. The context
releases its models on exit. `achew_many()` returns results in input order;
failed files return a `ChewFailure` with `ok=False`.

The runnable example processes the same sample again. Replace the path list
with your own documents for a larger batch.


In [ ]:
async with bibr.Chewer(memory="balanced") as chewer:
    if configured_path:
        batch = await chewer.achew_many([paper_path])
    else:
        with as_file(files("bibr").joinpath("data/sample_paper.pdf")) as sample_path:
            batch = await chewer.achew_many([sample_path])

for batch_result in batch:
    if batch_result.ok:
        print(f"  {batch_result.title}")
    else:
        print(f"  Failed {batch_result.path}: {batch_result.error}")


## 5. JSON export

`Result.save()` writes the export to disk for use from R, other services,
or Metacheck. This saves the single-paper result from the first example.


In [ ]:
import tempfile

output_path = Path(tempfile.gettempdir()) / "bibr_demo_export.json"
result.save(output_path)
print(f"Saved to {output_path} ({output_path.stat().st_size:,} bytes)")
